In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

# build vocabulary of characters and mappings to/from ints

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

# build dataset split into train / val / test

def build_dataset(words, block_size=3):
    X, Y = [], []

    for w in words:

        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)
    return X, Y

X, Y = build_dataset(words)

s1 = int(0.8 * len(X))
s2 = int(0.9 * len(X))

X_train, Y_train = X[:s1], Y[:s1]
X_val, Y_val = X[s1:s2], Y[s1:s2]
X_test, Y_test = X[s2:], Y[s2:]

print(f'train: {len(X_train)} val: {len(X_val)} test: {len(X_test)}')

In [ ]:
# utility function for comparing the manual and torch gradients
def cmp(s, dt, t):
    """
    s: name of the variable
    dt: manual gradient
    t: torch gradient
    """

    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approx: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
N_EMBD = 10
N_HIDDEN = 64
VOCAB_SIZE = len(stoi)
BLOCK_SIZE = 3
EPOCHS = 1_000
BATCH_SIZE = 32
LR = 0.135

g = torch.Generator().manual_seed(42)
C = torch.randn((VOCAB_SIZE, N_EMBD), generator=g)

# Layer 1
W1 = torch.randn((N_EMBD * BLOCK_SIZE, N_HIDDEN), generator=g) * (5/3) / ((N_EMBD * BLOCK_SIZE) ** 0.5)
b1 = torch.randn(N_HIDDEN, generator=g) * 0.1

# Layer 2
W2 = torch.randn((N_HIDDEN, VOCAB_SIZE), generator=g) * 0.1
b2 = torch.randn(VOCAB_SIZE, generator=g) * 0.1

# BatchNorm params
bngain = torch.ones((1, N_HIDDEN)) * 0.1 + 1.0
bnbias = torch.zeros((1, N_HIDDEN)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

total_params = sum(p.nelement() for p in parameters) # total number of parameters
print(f'Total parameters: {total_params}')

for p in parameters: p.requires_grad = True # enable gradients

In [ ]:
# constuct a minibatch
ix = torch.randint(0, X_train.shape[0], (BATCH_SIZE, ), generator=g)
Xb, Yb = X_train[ix], Y_train[ix]
print(Xb.shape, Yb.shape)

In [ ]:
# Forward propagation, divided into smaller parts, se we can compute backpropagation manually for every step

# Step 1: Embedding
emb = C[Xb] # embed the input character
embcat = emb.view(emb.shape[0], -1) # concatenate embeddings

# Step 2: Linear Layer 1
hprebn = embcat @ W1 + b1 # pre-batchnorm pre-activation hidden layer

# Step 3: BatchNorm
bnmeani = 1 / BATCH_SIZE * hprebn.sum(dim=0, keepdim=True)
bndiff_1 = hprebn - bnmeani
bndiff_2 = bndiff_1 ** 2
bnvar = 1 / (BATCH_SIZE- 1) * (bndiff_2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n - 1, not n)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff_1 * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Step 4: Non-linearity (tanh)
h = torch.tanh(hpreact) # hidden layer

# Step 5: Linear Layer 2
logits = h @ W2 + b2 # output layer

# Step 6: Cross-entropy loss (divided into smaller parts)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(BATCH_SIZE), Yb].mean()

# Step 7: PyTorch backward pass and retaining gradients for manual comparison
for p in parameters: p.grad = None # zero gradients

steps = [
    logprobs, probs, counts, counts_sum, counts_sum_inv,
    norm_logits, logit_maxes, logits, h, hpreact, bnraw,
    bnvar_inv, bnvar, bndiff_2, bndiff_1, bnmeani, hprebn,
    embcat, emb
] # order of steps in forward pass (from bottom to top)

for step in steps: step.retain_grad() # keep gradients for all steps

loss.backward()

print(f'loss: {loss.item()}')

In [ ]:
# Exercice 1: Embedding